In [3]:
import warnings
warnings.filterwarnings("ignore")

In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import copy
import gc
import os
import numpy as np
import pandas as pd

In [6]:
# Load the ultra-lightweight, instruction-following model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Detect if you have an active NVIDIA GPU
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    # If you have a GPU, load in Float16 (halves memory instantly)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    ).to(device)
else:
    # If running strictly on an <8GB RAM CPU
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    ).to(device)

print(f"Model successfully loaded on {device} without extra dependencies!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model successfully loaded on cpu without extra dependencies!


In [7]:
def rephrase_text_local(text):
    messages = [
        {
            "role": "system", 
            "content": "You are a renter searching for an apartment. Convert the listing description into a short, natural language search query a human would type. Do not use quotes, keep it short (max 12 words), and only output the query itself."
        },
        {
            "role": "user", 
            "content": text
        }
    ]
    
    templated_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([templated_text], return_tensors="pt").to(model.device)
    
    # Generate tokens safely
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, 
            max_new_tokens=30,
            temperature=0.6, 
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # Critical Low-RAM Clean Up: Clear cache and garbage collect variables
    del model_inputs, generated_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect() 
    
    return response.strip()

In [10]:
sample_listing = "This unit is located at 1717 12th Ave, Seattle, 98122, WAMonthly rental rates range from $925"
print("Input :", sample_listing)
print("Output:", rephrase_text_local(sample_listing))

Input : This unit is located at 1717 12th Ave, Seattle, 98122, WAMonthly rental rates range from $925
Output: What's the monthly rental rate for this unit located at 1717 12th Ave in Seattle, WA?


In [12]:
eval_dataset = [
    {'query_text': 'This unit is located at second St NE, Uhland Terrace NE, Washington, DC 20002, Washington, 20002, DC', 'max_price': 790, 'min_beds': 0, 'expected_ids': ['5668626895']},
    {'query_text': 'This unit is located at 814 Schutte Road, Evansville, 47712, INMonthly rental rates range from $425 ', 'max_price': 425, 'min_beds': 1, 'expected_ids': ['5664597177']},
    {'query_text': 'This unit is located at N Scott St, 14th St N, Arlington, VA 22209, Arlington, 22209, VAMonthly rent', 'max_price': 1390, 'min_beds': 0, 'expected_ids': ['5668626833']},
    {'query_text': 'This unit is located at 1717 12th Ave, Seattle, 98122, WAMonthly rental rates range from $925We have', 'max_price': 925, 'min_beds': 0, 'expected_ids': ['5659918074']},
    {'query_text': 'This unit is located at Washington Blvd, N Cleveland St, Arlington, Arlington, 22201, VAMonthly rent', 'max_price': 880, 'min_beds': 0, 'expected_ids': ['5668626759']},
    {'query_text': '**RARE GEM WITH PRIVATE OUTDOOR TERRACE****AVAILABLE IMMEDIATELY** $2475 RENT IS NET EFFECTIVE WITH ', 'max_price': 2475, 'min_beds': 0, 'expected_ids': ['5667891676']},
    {'query_text': 'This unit is located at 2432 Penmar Ave, Venice, 90291, CAMonthly rental rates range from $1800We ha', 'max_price': 1800, 'min_beds': 0, 'expected_ids': ['5668627426']},
    {'query_text': 'This unit is located at Oak St NW, 16th St NW, Washington, DC 20010, Washington, 20010, DCMonthly re', 'max_price': 840, 'min_beds': 0, 'expected_ids': ['5668626687']},
    {'query_text': 'This unit is located at 333 Hyde St, San Francisco, 94109, CAMonthly rental rates range from $1495We', 'max_price': 1495, 'min_beds': 0, 'expected_ids': ['5668610290']},
    {'query_text': 'This unit is located at A St SE, 19th St SE, Washington, Washington, 20003, DCMonthly rental rates r', 'max_price': 890, 'min_beds': 0, 'expected_ids': ['5668627023']},
    {'query_text': 'This unit is located at 15th St SE, Independence Ave SE, Washington DC, Washington, 20003, DCMonthly', 'max_price': 990, 'min_beds': 0, 'expected_ids': ['5668627099']},
    {'query_text': 'This unit is located at Arkansas Ave NW, Varnum St NW, Washington, Washington, 20011, DCMonthly rent', 'max_price': 840, 'min_beds': 0, 'expected_ids': ['5668626548']},
    {'query_text': 'This unit is located at 2326 N sixth Avenue, Tucson, 85705, AZMonthly rental rates range from $465 -', 'max_price': 1795, 'min_beds': 1, 'expected_ids': ['5664598162']},
    {'query_text': 'This unit is located at Salem Ln NW, 45th St NW, Washington, Washington, 20007, DCMonthly rental rat', 'max_price': 1090, 'min_beds': 0, 'expected_ids': ['5668626900']},
    {'query_text': 'This unit is located at 57 Taylor Street, San Francisco, 94102, CAMonthly rental rates range from $1', 'max_price': 1695, 'min_beds': 0, 'expected_ids': ['5664571820']},
    {'query_text': 'New Bern Studio includes : 1 beds, 1 microwave, 1 mini fridge and a bath room. Cottages have linens,', 'max_price': 1560, 'min_beds': 1, 'expected_ids': ['5659276240']},
    {'query_text': 'New Bern Studio includes : 1 bedrooms, 1 microwave, one mini fridge and a bath room. Cottages have l', 'max_price': 1560, 'min_beds': 1, 'expected_ids': ['5654898031']},
    {'query_text': 'This unit is located at Spring Ridge Dr, Spring, 77386, TXMonthly rental rates range from $1000We ha', 'max_price': 1000, 'min_beds': 1, 'expected_ids': ['5664574876']},
    {'query_text': 'This unit is located at 545 Georgia street 717-723 Sutter Street, Vallejo, 94590, CAMonthly rental r', 'max_price': 950, 'min_beds': 1, 'expected_ids': ['5668633573']},
    {'query_text': 'This unit is located at Falcon Wood Dr, Marietta, 30066, GAMonthly rental rates range from $625We ha', 'max_price': 625, 'min_beds': 1, 'expected_ids': ['5668624220']},
    {'query_text': 'This unit is located at Branthurst Dr, Charlotte, 28269, NCMonthly rental rates range from $600We ha', 'max_price': 600, 'min_beds': 1, 'expected_ids': ['5653564493']},
    {'query_text': 'This unit is located at 501 Chapel Drive, Tallahassee, 32304, FLMonthly rental rates range from $544', 'max_price': 544, 'min_beds': 1, 'expected_ids': ['5668622178']},
    {'query_text': 'This unit is located at Leeward Ct, Fleming Island, 32003, FLMonthly rental rates range from $525We ', 'max_price': 525, 'min_beds': 1, 'expected_ids': ['5664567303']},
    {'query_text': 'This unit is located at Union Hills, Phoenix, 85027, AZMonthly rental rates range from $450We have 1', 'max_price': 450, 'min_beds': 1, 'expected_ids': ['5664577209']},
    {'query_text': 'This unit is located at Herons Path Pl, Riverview, 33578, FLMonthly rental rates range from $750We h', 'max_price': 750, 'min_beds': 1, 'expected_ids': ['5664574105']},
    {'query_text': 'This unit is located at S. Pond Court, Lafayette, 94549, ALMonthly rental rates range from $2000We h', 'max_price': 2000, 'min_beds': 1, 'expected_ids': ['5668624126']},
    {'query_text': 'This unit is located at Arballo Dr 12d, San Francisco, 94132, CAMonthly rental rates range from $140', 'max_price': 1400, 'min_beds': 1, 'expected_ids': ['5668624017']},
    {'query_text': 'This unit is located at Elias Way, Glen Burnie, 21060, MDMonthly rental rates range from $1095We hav', 'max_price': 1095, 'min_beds': 1, 'expected_ids': ['5668624367']},
    {'query_text': 'This unit is located at W. Ken Caryl Place #d, Littleton, 80128, COMonthly rental rates range from $', 'max_price': 915, 'min_beds': 1, 'expected_ids': ['5664570217']},
    {'query_text': 'This unit is located at Skipjack Ct, Waldorf, 20603, MDMonthly rental rates range from $900We have 1', 'max_price': 900, 'min_beds': 1, 'expected_ids': ['5664574791']}
]



In [15]:
robust_eval_dataset = copy.deepcopy(eval_dataset)

for idx, item in enumerate(robust_eval_dataset):
    original = item['query_text']
    rephrased = rephrase_text_local(original)
    item['query_text'] = rephrased
    print(f"[{idx+1}/{len(robust_eval_dataset)}] -> {rephrased}")

print("--- Paraphrasing Finished ---")

[1/30] -> What apartment is located at 2nd St NE in Washington, DC?
[2/30] -> What's the monthly rental rate for this unit at 814 Schutte Road in Evansville, Indiana?
[3/30] -> N Scott St, 14th St N, Arlington, VA 22209, Arlington, 22209,
[4/30] -> What's the monthly rental rate for this apartment located at 1717 12th Ave in Seattle?
[5/30] -> Washington Blvd, N Cleveland St, Arlington, Arlington, 22201, VAMonthly Rent
[6/30] -> RARE GEM with private outdoor terrace, available immediately, net effective with 2475 rental cost
[7/30] -> What's the monthly rental rate for this unit located at 2432 Penmar Ave in Venice, California?
[8/30] -> What apartment is located at Oak St NW, 16th St NW in Washington, DC?
[9/30] -> What is the rental rate for this unit located at 333 Hyde St in San Francisco, CA?
[10/30] -> What's the rent in this apartment?
[11/30] -> What apartment is located at 15th St SE in Independence Ave SE, Washington DC?
[12/30] -> Where is the apartment located on Arkansas A

In [17]:
# Convert the list of dicts directly into a structured Pandas DataFrame
df_eval = pd.DataFrame(robust_eval_dataset)

# Clean up the 'expected_ids' list column to store cleanly as a standard string/comma-separated value
# This makes it easier to read inside spreadsheet programs like Excel or Sheets
df_eval['expected_ids'] = df_eval['expected_ids'].apply(lambda x: ",".join(x))

output_filename = "eval_dataset.csv"
df_eval.to_csv(output_filename, index=False)

print(f"Successfully saved {len(df_eval)} queries to: {output_filename}")

Successfully saved 30 queries to: eval_dataset.csv
